# Pose-Controlled Image Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tigermorning/pose-image-tool/blob/main/pose_tool.ipynb)

Generate images that follow **both** a reference pose and a text prompt.

**Pipeline:** reference photo -> OpenPose keypoints + MiDaS depth -> two hint images -> two ControlNets -> SDXL -> output image.

| | |
|---|---|
| Base model | `SG161222/RealVisXL_V5.0` - a photoreal SDXL fine-tune |
| ControlNet - pose | `thibaud/controlnet-openpose-sdxl-1.0` |
| ControlNet - depth | `diffusers/controlnet-depth-sdxl-1.0` |
| Annotators | `controlnet_aux` `OpenposeDetector` + `MidasDetector` (`lllyasviel/Annotators`) |
| GPU | Colab free **T4** is enough (fp16, ~10 GB peak). Placement adapts to the VRAM and RAM the notebook measures. |

**Before you start:** `Runtime > Change runtime type > T4 GPU`, and have 2 photos of people in clearly different poses ready to upload (section 3).

## How this notebook answers the assignment

| Assignment | Where | Output |
|---|---|---|
| **Step 1** - one reference photo, extract the pose, generate a *different person* in that same pose | section 5, prompt **A1** | `samples/output_01.png` |
| **Step 2a** - same pose, change only the prompt | section 5, **A1 vs A2** | `output_01.png`, `output_01_alt_prompt.png` |
| **Step 2b** - same prompt, change only the pose photo | section 5, **B1 vs B2** | `output_02_pose01.png`, `output_02.png` |
| **Step 2** write-up - what was changed, how the output changed | section 7 | - |

`output_02_pose01.png` and `output_02.png` share prompt and seed byte for byte and differ only in
which photograph the skeleton came from. That difference is the evidence the tool works.

Step 2b's prompt is **A1 with its three posture clauses removed**, not A1 verbatim: A1 describes
`pose_01` specifically, so sending it unchanged to a standing reference would vary two things at
once. See `prompts.md`.

**Sections:** 1 Install dependencies - 2 Load models - 3 Extract pose - 4 Generate image - 5 Experiments - 6 Save results - 7 Findings

---
## 1. Install dependencies

In [ ]:
# Install the diffusion stack and the OpenPose annotator.
# controlnet_aux is pinned to 0.0.10: 0.0.9 constrains timm so tightly that pip cannot resolve it.
# Do NOT add mediapipe. controlnet_aux imports it at package level and against Colab's build that
# raises `AttributeError: module 'mediapipe' has no attribute 'solutions'`, which breaks the import
# of OpenposeDetector itself. Uninstalled, it only prints a warning and everything works.
!pip install -q "diffusers>=0.31" "transformers>=4.44" "accelerate>=0.34" safetensors
!pip install -q "controlnet_aux==0.0.10"

In [ ]:
# Confirm a CUDA GPU is attached, then measure BOTH memory ceilings before anything loads.
# Colab enforces them separately and they fail differently: exhausting system RAM kills the
# session outright, exhausting VRAM raises CUDA out of memory. The placement decision in the
# next cell reads these numbers, so they are measured rather than assumed.
import platform
import shutil

import torch

assert torch.cuda.is_available(), 'No GPU attached. Colab: Runtime > Change runtime type > T4 GPU.'
_props = torch.cuda.get_device_properties(0)
FREE_VRAM_GB, TOTAL_VRAM_GB = (b / 1024 ** 3 for b in torch.cuda.mem_get_info())

try:
    import psutil

    FREE_RAM_GB = psutil.virtual_memory().available / 1024 ** 3
    TOTAL_RAM_GB = psutil.virtual_memory().total / 1024 ** 3
except ImportError:              # psutil ships with Colab, but do not hard-fail without it
    FREE_RAM_GB = TOTAL_RAM_GB = float('nan')

print(f'python : {platform.python_version()}')
print(f'torch  : {torch.__version__}')
print(f'gpu    : {_props.name}')
print(f'vram   : {FREE_VRAM_GB:.1f} GB free of {TOTAL_VRAM_GB:.1f} GB')
print(f'ram    : {FREE_RAM_GB:.1f} GB free of {TOTAL_RAM_GB:.1f} GB')
print(f'disk   : {shutil.disk_usage(".").free / 1024 ** 3:.1f} GB free')

# T4 has no usable bf16 path, so fp16 is used for every module in this notebook.
DTYPE = torch.float16

---
## 2. Load models

Three cells. The first two load models, the third decides where they live.

1. **OpenPose annotator** - finds human keypoints. It only *reads* images; it generates nothing.
2. **SDXL + ControlNet** - generates images. ControlNet is the adapter that injects the skeleton
   into every denoising step.
3. **Placement** - keeps everything on the GPU when it fits, and offloads only when it does not.

In [ ]:
# MODEL LOAD 1 of 2 - the two annotators.
# Neither generates anything. They read the reference photo and produce the two hints:
# OpenPose gives joint coordinates, MiDaS gives a depth map.
import numpy as np
from controlnet_aux import MidasDetector, OpenposeDetector
# drawing helpers, used to render the skeleton at the exact generation resolution
from controlnet_aux.open_pose.util import draw_bodypose, draw_handpose

# Guarded so re-running this cell does not load a second copy alongside the first.
if 'openpose' not in globals():
    openpose = OpenposeDetector.from_pretrained('lllyasviel/Annotators')
if 'midas' not in globals():
    midas = MidasDetector.from_pretrained('lllyasviel/Annotators')
print('annotators ready')

In [ ]:
# MODEL LOAD 2 of 2 - the image generator.
# Three pieces: a photoreal SDXL checkpoint, and TWO ControlNets that steer it together.
#
#   pose  - an OpenPose skeleton. Says where the joints are.
#   depth - a depth map. Says which way the torso leans, what the body rests on, how the limbs
#           sit in space. A skeleton carries none of that, and this pose depends on it.
#
# The base is RealVisXL rather than SDXL base 1.0. Measured on this reference: base 1.0 rendered
# faces and skin that read as a mannequin, and no prompt or sampler setting fixed it, because it
# is the checkpoint's own character. Swapping the checkpoint did fix it.
from diffusers import AutoencoderKL, ControlNetModel, StableDiffusionXLControlNetPipeline

BASE_ID = 'SG161222/RealVisXL_V5.0'
POSE_CN_ID = 'thibaud/controlnet-openpose-sdxl-1.0'
DEPTH_CN_ID = 'diffusers/controlnet-depth-sdxl-1.0'

if 'pipe' in globals():
    print('pipeline already loaded - skipping')
else:
    pose_cn = ControlNetModel.from_pretrained(POSE_CN_ID, torch_dtype=DTYPE)
    depth_cn = ControlNetModel.from_pretrained(DEPTH_CN_ID, torch_dtype=DTYPE)
    vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=DTYPE)
    pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
        BASE_ID,
        controlnet=[pose_cn, depth_cn],
        vae=vae,
        torch_dtype=DTYPE,
        use_safetensors=True,
    )
    print('models loaded, not yet placed on a device')

In [ ]:
# DECIDE WHERE THE MODELS LIVE, from the memory measured in section 1.
# The trap worth knowing: offloading trades VRAM for system RAM, so on a RAM-limited machine it
# makes the more dangerous failure - the one that kills the session outright - more likely rather
# than less. A 15 GB T4 holds the whole pipeline, so there the fast path is also the safe one.
import gc

# Rough fp16 footprint at 832x1216: UNet ~5.1 GB, TWO ControlNets ~5.0 GB, text encoders
# ~1.5 GB, VAE ~0.2 GB, plus activations. 13 GB is that total with headroom.
VRAM_NEEDED_GB = 13.0
RAM_NEEDED_FOR_OFFLOAD_GB = 8.0

if FREE_VRAM_GB >= VRAM_NEEDED_GB:
    pipe.to('cuda')
    placement = f'full GPU ({FREE_VRAM_GB:.1f} GB VRAM free, nothing held in system RAM)'
elif FREE_RAM_GB >= RAM_NEEDED_FOR_OFFLOAD_GB:
    # Whole modules move to the GPU as each is called, then back. Slower than resident, but far
    # faster than sequential offload, which the diffusers docs call extremely slow.
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_tiling()       # decodes the latent in overlapping tiles: lowers peak VRAM
    placement = f'model CPU offload ({FREE_VRAM_GB:.1f} GB VRAM, {FREE_RAM_GB:.1f} GB RAM free)'
else:
    # Both ceilings are tight. Sequential offload moves submodules rather than whole models: the
    # smallest VRAM footprint available, and by far the slowest.
    pipe.enable_sequential_cpu_offload()
    pipe.enable_vae_tiling()
    placement = 'sequential CPU offload - LAST RESORT, expect minutes per image'

# No enable_vae_slicing() here on purpose. Slicing splits a *batch* of latents, and the diffusers
# docs state there is no performance impact for single-image batches, which is all this notebook
# generates. Tiling is the setting that lowers peak memory for one large image.
pipe.set_progress_bar_config(leave=False)
gc.collect()
torch.cuda.empty_cache()
print(f'pipeline ready - {placement}')

---
## 3. Extract pose

Upload the reference photos, then convert each one into a skeleton image.

- **Experiment A** (same pose, different prompts) needs **1** reference.
- **Experiment B** (same prompt, different poses) needs **2+** references with visibly different poses.

Good references: one person, full body or at least head-to-knee, limbs not overlapping the torso. Crowds and heavy occlusion are where OpenPose fails.

In [ ]:
# Upload the reference photos (jpg / png / webp). They are kept in ./references,
# and every artifact this notebook produces goes to ./samples.
from pathlib import Path

from PIL import Image, ImageOps

REF_DIR = Path('references')
OUT_DIR = Path('samples')
REF_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

try:
    from google.colab import files

    for fname, data in files.upload().items():
        (REF_DIR / fname).write_bytes(data)
except ImportError:
    print('Not running on Colab - copy your images into ./references by hand, then re-run this cell.')

SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}
REFS = sorted(p for p in REF_DIR.iterdir() if p.suffix.lower() in SUFFIXES)
assert REFS, 'No reference images found in ./references'
print(f'{len(REFS)} reference image(s):', [p.name for p in REFS])

In [ ]:
# Helpers. The ControlNet hint and the generated image must have identical dimensions,
# so every reference is cropped to one fixed portrait resolution up front.
WIDTH, HEIGHT = 832, 1216  # SDXL-native portrait ratio; lighter on a T4 than 1024x1024


def load_ref(path, width=WIDTH, height=HEIGHT):
    """Open a reference photo, apply its EXIF rotation, and letterbox it to the target ratio.

    Letterboxing, not centre-cropping. A crop silently removes whatever falls outside the target
    aspect ratio: on one of the references here it cut the left ankle out of frame, which cost a
    keypoint and disabled pose control for that leg. Padding keeps the whole body.
    """
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    return ImageOps.pad(img, (width, height), method=Image.LANCZOS)

In [ ]:
# THE POSE EXTRACTION STEP. Turns one reference photo into the two hints ControlNet reads.
def extract_hints(image, include_hand=True):
    """Return (skeleton, depth) for one reference, both at the generation resolution.

    The skeleton is drawn from keypoints rather than taken from OpenposeDetector.__call__, which
    renders at its own resolution and rescales - that blurs the stick figure and changes its
    aspect ratio, and a blurred hint made SDXL draw three legs earlier in this project.
    """
    width, height = image.size
    people = openpose.detect_poses(np.array(image), include_hand=include_hand, include_face=False)
    if not people:
        raise RuntimeError('OpenPose found no person in this image.')

    canvas = np.zeros((height, width, 3), dtype=np.uint8)
    person = people[0]
    canvas = draw_bodypose(canvas, person.body.keypoints)
    if include_hand:
        canvas = draw_handpose(canvas, person.left_hand)
        canvas = draw_handpose(canvas, person.right_hand)

    depth = midas(image, detect_resolution=512, image_resolution=min(image.size))
    depth = depth.convert('RGB').resize((width, height), Image.BICUBIC)
    return Image.fromarray(canvas), depth

In [ ]:
# RUN THE EXTRACTION over every uploaded reference, and save both hints.
poses = []
for i, path in enumerate(REFS, start=1):
    ref = load_ref(path)
    skeleton, depth = extract_hints(ref)
    skeleton.save(OUT_DIR / f'pose_{i:02d}.png')
    depth.save(OUT_DIR / f'depth_{i:02d}.png')
    poses.append({'id': f'{i:02d}', 'source': path.name, 'ref': ref,
                  'pose': skeleton, 'depth': depth})

print('saved:', [f"pose_{p['id']}.png + depth_{p['id']}.png" for p in poses])

### 3a. Check the hints before generating

**Do not skip this.** An unclosed hip-to-ankle chain silently disables pose control for that limb, and nothing later in the run reports it.

In [ ]:
# Check each hint NUMERICALLY before spending GPU time on it.
# A skeleton can look plausible and still be unusable: if a hip-to-ankle chain does not close,
# pose control over that limb is silently off and nothing later in the run will say so. Four of
# six candidate references for this notebook were rejected by exactly this check.
COCO18 = ['nose', 'neck', 'r_shoulder', 'r_elbow', 'r_wrist', 'l_shoulder', 'l_elbow', 'l_wrist',
          'r_hip', 'r_knee', 'r_ankle', 'l_hip', 'l_knee', 'l_ankle', 'r_eye', 'l_eye',
          'r_ear', 'l_ear']
CHAINS = {'right arm': (2, 3, 4), 'left arm': (5, 6, 7),
          'right leg': (8, 9, 10), 'left leg': (11, 12, 13)}


def validate_pose(image, label=''):
    """Print the keypoint count and any limb chain that failed to close."""
    people = openpose.detect_poses(np.array(image))
    if len(people) != 1:
        print(f'{label}: FAIL - {len(people)} people detected, this pipeline assumes exactly 1')
        return False

    keypoints = people[0].body.keypoints
    found = {i for i, kp in enumerate(keypoints) if kp is not None}
    broken = [name for name, chain in CHAINS.items() if not set(chain) <= found]

    print(f'{label}: {len(found)}/18 keypoints')
    if broken:
        missing = ', '.join(COCO18[i] for i in range(18) if i not in found)
        print(f'  INCOMPLETE: {", ".join(broken)}  (missing {missing})')
        print('  -> pose control is off for those limbs; replace this reference')
    else:
        print('  all four limb chains complete')
    return not broken


for entry in poses:
    entry['valid'] = validate_pose(entry['ref'], f"pose_{entry['id']} ({entry['source']})")

In [ ]:
# Inspect every reference next to its skeleton. Do this before generating anything:
# a missing arm or a merged pair of legs here will be a missing arm in the output too.
import matplotlib.pyplot as plt
import numpy as np


def show_grid(images, titles, cols=None, size=3.0):
    """Display images in a labelled grid (used for every preview and experiment below)."""
    cols = cols or len(images)
    rows = -(-len(images) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * 1.45 * rows), squeeze=False)
    flat = axes.ravel()
    for ax, img, title in zip(flat, images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
    for ax in flat:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close(fig)   # otherwise every call leaves a figure open


show_grid(
    [im for p in poses for im in (p['ref'], p['pose'], p['depth'])],
    [t for p in poses for t in (p['source'], f"pose_{p['id']}", f"depth_{p['id']}")],
    cols=3,
)

---
## 4. Generate image

One helper does all generation. Every experiment below only changes its arguments, so the variable under test is always explicit.

Key knobs:

| Argument | Effect |
|---|---|
| `pose_scale` | How hard the skeleton is enforced. Low = prompt wins, high = pose wins but anatomy stiffens. |
| `depth_scale` | How hard the depth map is enforced. Carries torso lean, contact and limb ordering. |
| `guidance_scale` | How hard the text prompt is enforced. |
| `seed` | Fixed by default so experiments stay single-variable. |

The two scales are not equal. Pose defaults to **0.8** and depth to **0.6**: depth already pins the limbs, so giving pose full weight on top of it stiffens the body. **Diagnostic 2**, after the experiments, re-measures that against pose-only conditioning in this run rather than asking you to take it on trust. On a pose-only pipeline the opposite held - there 0.8 let a folded knee land 0.45 of the frame from the hint and 1.0 was required, which is a figure from a configuration this notebook no longer runs.

In [ ]:
# THE IMAGE GENERATION STEP.
# Conditions on three things at once: the skeleton, the depth map, and the text prompt.
#
# The two conditioning scales are not equal. Pose is held slightly lower than depth would suggest
# because depth already pins the limbs; giving pose full weight on top of depth stiffens the body.
# Diagnostic 2, after the experiments, re-measures this against pose-only conditioning in the same
# run, so the justification is not a number carried in from somewhere else.
NEGATIVE = ('doll, mannequin, plastic skin, waxy, airbrushed, lowres, blurry, deformed hands, '
            'extra fingers, extra limbs, three legs, duplicated limbs, fused limbs, watermark, text')


def generate(hints, prompt, seed=1234, steps=28, guidance_scale=5.0,
             pose_scale=0.8, depth_scale=0.6, negative_prompt=NEGATIVE):
    """Return one image following the skeleton, the depth map and the prompt together.

    `hints` is the (skeleton, depth) pair from extract_hints(). The seed is explicit so any two
    calls can be compared as a controlled experiment.
    """
    torch.cuda.empty_cache()   # keep VRAM from fragmenting across a run
    skeleton, depth = hints
    return pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=[skeleton, depth],
        width=skeleton.width,
        height=skeleton.height,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        controlnet_conditioning_scale=[pose_scale, depth_scale],
        generator=torch.Generator('cuda').manual_seed(seed),
    ).images[0]

### 4a. Score the skeleton match

**Diagnostic, and narrower than it looks.** It compares joint coordinates only, so it catches scattered or duplicated limbs. Torso lean, what the hands are touching and where the weight sits are not in a skeleton and are not measured here - judge those by eye against the reference photograph.

In [ ]:
# Score how closely the generated image's skeleton matches the hint.
#
# Read this number for what it is. It compares eighteen joint COORDINATES, so it catches limbs
# that scattered or duplicated. It cannot see which way the torso leans, what the hands are
# touching, or where the body's weight sits - none of that is in a skeleton. A generated image
# can score 0.02 here and still read as a different pose to a person. Use it to rule out tangled
# limbs, and judge the pose itself by eye against the reference photograph.
def pose_fidelity(hint_keypoints, image, label=''):
    """Report mean and max joint displacement between the hint and the generated image.

    Distances are normalised to the image (0.03 is a close match, 0.3 is a different pose).
    Only the limbs are scored; head keypoints follow the prompt more than the hint.
    """
    detected = openpose.detect_poses(np.array(image))
    if not detected:
        print(f'{label}: no person detected in the output')
        return None

    got = detected[0].body.keypoints
    scored, worst = [], ('', 0.0)
    for i in (2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13):
        a, b = hint_keypoints[i], got[i]
        if a is None or b is None:
            continue
        distance = ((a.x - b.x) ** 2 + (a.y - b.y) ** 2) ** 0.5
        scored.append(distance)
        if distance > worst[1]:
            worst = (COCO18[i], distance)
    if not scored:
        print(f'{label}: no comparable joints')
        return None
    mean = sum(scored) / len(scored)
    print(f'{label}: mean joint error {mean:.3f}, worst {worst[0]} {worst[1]:.3f} '
          f'({len(scored)} joints)')
    return mean


# Keypoints of each hint, kept so generated images can be scored against them.
for entry in poses:
    detected = openpose.detect_poses(np.array(entry['ref']))
    entry['keypoints'] = detected[0].body.keypoints if detected else None
print('hint keypoints stored for fidelity scoring')

---
## 5. Experiments - the assignment's step 1 and step 2

Two runs, each isolating one variable. Prompt A1 doubles as step 1's deliverable: it is the
*different person in the reference's pose*. Prompts, seeds and settings are recorded in
`prompts.md`.

### Step 1 and Step 2a - same pose, different prompts

**A1 is step 1**: the reference is a seated man, and this prompt asks for a young African woman. Whatever posture survives comes from the skeleton alone, because the prompt never mentions one.

**A1 against A2 is step 2a**: the hint, the seed and every sampler setting are held fixed, so any difference between the two images is attributable to the prompt text and nothing else.

In [ ]:
# EXPERIMENT A - one pose, two prompts, identical seed and sampler settings.
# A1 is the assignment's step 1: a different person in the reference's pose.
#
# Both prompts describe what the hints cannot: which way the torso leans, where the weight sits,
# what the hands are touching. The depth map carries much of this, the prompt reinforces it.
# What still must stay out of a prompt is anything the skeleton already fixes - limb angles.
HINTS_A = (poses[0]['pose'], poses[0]['depth'])
SEED = 1234                     # one seed throughout, so every comparison is single-variable
PROMPTS_A = [
    'A beautiful young African woman perched on the edge of a metal stool, leaning forward with her weight on her arm, one hand gripping the seat between her knees, wide dark trousers and a cropped jacket, sunlit concrete gallery, 85mm lens at f/2, candid editorial photograph, natural skin with visible pores',
    'An ink and watercolour drawing of a travelling puppeteer perched on the edge of a wooden stool, leaning forward with his weight on his arm, one hand gripping the seat between his knees, patched wool coat and worn boots, cream paper with visible fibres, loose brush strokes, ochre and burnt umber',
]

exp_a = []
for i, prompt in enumerate(PROMPTS_A, start=1):
    image = generate(HINTS_A, prompt, seed=SEED)
    exp_a.append(image)
    pose_fidelity(poses[0]['keypoints'], image, f'A{i}')

show_grid([poses[0]['pose'], poses[0]['depth'], *exp_a],
          [f"pose_{poses[0]['id']}", f"depth_{poses[0]['id']}", 'A1', 'A2'])

### Step 2b - same prompt, different pose photo

The prompt and the seed are fixed. The only thing that changes is which photograph the skeleton was extracted from: `pose_01` (seated on a stool) or `pose_02` (standing, fists raised). This is the inverse of step 2a.

`PROMPT_B` is **A1 with its three posture clauses deleted** and nothing else changed - byte-identical from `wide dark trousers` onward. It is not A1 verbatim, because A1 describes `pose_01` (`perched on the edge of a metal stool`, and so on) and those clauses contradict a standing reference. Holding a prompt that fits only one of the two hints would vary two things at once. It also gives the run one prompt that names no posture at all, which is what proves the pose came from the skeleton rather than from the text.

In [ ]:
# EXPERIMENT B - one prompt, two references, identical seed and sampler settings.
# This prompt is deliberately pose-neutral and therefore not A1: it goes to a seated reference and
# a standing one, so a posture clause would contradict one of them.
assert len(poses) >= 2, 'Upload at least two references with different poses to run experiment B.'
PROMPT_B = 'A beautiful young African woman, wide dark trousers and a cropped jacket, sunlit concrete gallery, 85mm lens at f/2, candid editorial photograph, natural skin with visible pores'

exp_b = []
for entry in poses[:2]:
    image = generate((entry['pose'], entry['depth']), PROMPT_B, seed=SEED)
    pose_fidelity(entry['keypoints'], image, f"B on pose_{entry['id']}")
    exp_b.append(image)

used = poses[:len(exp_b)]
show_grid([e['pose'] for e in used] + exp_b,
          [f"pose_{e['id']}" for e in used] + [f"B on pose_{e['id']}" for e in used],
          cols=len(used))

### Diagnostics - the two controls section 7 argues from

Neither of these is a deliverable, and neither is saved to `samples/`. They exist because section 7
makes two claims that would otherwise rest on numbers measured somewhere else, on different
hardware or on a pipeline this notebook no longer runs. Both are re-measured here, in this run, on
this GPU, so every figure quoted in the write-up comes from one place.

1. **A posture clause that contradicts the hint is dropped, not blended.** A1 describes `pose_01`
   (`perched on the edge of a metal stool`). Sent unchanged to the standing `pose_02`, does the
   pose degrade? Compare against `B on pose_02`, which is the same hint with the posture clauses
   removed.
2. **Pose 0.8 with depth 0.6 beats pose alone at 1.0.** Setting `depth_scale=0.0` disables the
   depth ControlNet's contribution, leaving pose-only conditioning. Compare against `A1`, which is
   the same prompt, hint and seed at the notebook's defaults.

In [ ]:
# DIAGNOSTIC 1 - does a contradicting posture clause survive?
# A1 verbatim over the standing hint. A1's posture clauses describe the seated reference, so they
# fight this one. Not saved: this is a control, not a deliverable.
entry_standing = poses[1]
diag_conflict = generate((entry_standing['pose'], entry_standing['depth']), PROMPTS_A[0], seed=SEED)
score_conflict = pose_fidelity(entry_standing['keypoints'], diag_conflict, 'A1 over pose_02 (clauses fight the hint)')

show_grid([entry_standing['pose'], exp_b[1], diag_conflict],
          [f"pose_{entry_standing['id']}", 'B on pose_02 (no posture clauses)', 'A1 over pose_02'])
print('
Compare the two outputs against each other, not just the scores. The question is whether')
print('the stool, the forward lean and the hand on the seat appear where the prompt asked for them.')

In [ ]:
# DIAGNOSTIC 2 - is pose 0.8 + depth 0.6 actually better than pose alone at 1.0?
# depth_scale=0.0 zeroes the depth ControlNet's residuals, leaving pose-only conditioning.
# Everything else - prompt, hint, seed, steps, guidance - is held. Not saved: a control.
diag_pose_only = generate(HINTS_A, PROMPTS_A[0], seed=SEED, pose_scale=1.0, depth_scale=0.0)
score_pose_only = pose_fidelity(poses[0]['keypoints'], diag_pose_only, 'A1 pose-only at 1.0')

show_grid([poses[0]['pose'], poses[0]['depth'], exp_a[0], diag_pose_only],
          [f"pose_{poses[0]['id']}", f"depth_{poses[0]['id']}",
           'A1 (pose 0.8 + depth 0.6)', 'A1 (pose 1.0, no depth)'])
print('
The joint score only compares coordinates. Depth carries lean, contact and limb ordering,')
print('so also check whether the figure still rests on the stool rather than floating in front of it.')

---
## 6. Save results

Writes the four filenames the repository expects, then zips `samples/` for download so the images can be committed.

In [ ]:
# THE RESULT SAVING STEP.
# Writes only files that carry information nothing else does. The skeletons were saved in
# section 3; these are the generated images, named for the assignment step each answers.
import shutil

exp_a[0].save(OUT_DIR / 'output_01.png')             # step 1: a different person, the same pose
exp_a[1].save(OUT_DIR / 'output_01_alt_prompt.png')  # step 2a: same pose, the other prompt
exp_b[0].save(OUT_DIR / 'output_02_pose01.png')      # step 2b, seated arm
exp_b[1].save(OUT_DIR / 'output_02.png')             # step 2b, standing arm

shutil.make_archive('samples', 'zip', root_dir='.', base_dir='samples')  # cross-platform, no shell
print('samples/:', sorted(p.name for p in OUT_DIR.iterdir()))

# What the run actually cost, so the next person knows whether they had headroom.
print(f'peak VRAM: {torch.cuda.max_memory_allocated() / 1024 ** 3:.1f} GB')
try:
    import psutil

    print(f'RAM still free: {psutil.virtual_memory().available / 1024 ** 3:.1f} GB')
except ImportError:
    pass

try:
    from google.colab import files as colab_files

    colab_files.download('samples.zip')
except ImportError:
    print('samples.zip written next to the notebook')

---
## 7. Findings

Everything below was measured with the pipeline exactly as this notebook now runs it: the hint drawn
at the generation resolution, references letterboxed rather than cropped, hand keypoints enabled,
`controlnet_conditioning_scale` 1.0, `guidance_scale` 5.0, 28 steps, seed 1234, 832x1216.

Earlier drafts of this notebook had three defects that produced badly wrong images - a blurred and
vertically squashed hint, a centre crop that removed keypoints, and a conditioning scale too low to
hold a folded limb. Those outputs are not part of this repository and nothing here describes them as
results. They appear below only where the fix is worth documenting.

### What was changed

| Run | Held fixed | Varied |
|---|---|---|
| **Step 1 / 2a** | pose hint `pose_01`, seed 1234, 28 steps, guidance 5.0, conditioning 1.0 | the prompt: a photographic African-woman brief, then an ink-and-watercolour brief |
| **Step 2b** | the prompt (step 1's, verbatim), seed 1234, every sampler setting | the pose hint: `pose_01` seated, `pose_02` standing |

Both hints passed section 3a at 18/18 keypoints with all four limb chains closed and no limbs
crossing in 2D. That is the first reference pair in this project to have no defect at all; four
earlier candidates were rejected by the same check.

### What changed in the output

Each figure is the `pose_fidelity()` score printed by section 5: the pose re-detected in the
generated image and compared with the hint joint by joint, over the twelve limb joints. Roughly 0.03
means the pose was reproduced, 0.3 means a different pose.

| Output | Mean joint error | Worst joint | What it shows |
|---|---|---|---|
| `output_01.png` | **0.022** | r_ankle 0.042 | step 1 - a different person in the reference's pose |
| `output_01_alt_prompt.png` | 0.046 | r_ankle 0.160 | step 2a - same hint and seed, different prompt |
| `output_02.png` | 0.045 | r_knee 0.144 | step 2b - same prompt and seed, different hint |

**Step 1 worked.** `output_01.png` reproduces the reference pose to within 0.042 on every one of the
twelve joints - one leg extended to the ground, the other knee raised, both feet visible in sandals,
one hand at the hip and the other resting on the leg, the head turned as in the photograph. The
subject is a different person entirely. That is the claim the assignment asks for, and it is legible
without needing the score.

**Step 2a - changing only the prompt.** The hint, the seed and every sampler setting were held. Both
outputs keep the posture; everything the skeleton does not encode moved completely - person,
wardrobe, setting, palette, and medium, since the second prompt left photography for ink and
watercolour. The pose survives the medium change at 0.046, worse than 0.022 but still clearly the
same posture. **Moving away from photorealism costs pose fidelity without breaking it.**

**Step 2b - changing only the pose photo.** Identity, wardrobe and palette carried across both
images while posture followed whichever photograph the skeleton came from: seated in `output_01.png`,
standing with the forearms raised in `output_02.png`. Since prompt and seed are byte-identical
between those two files, the difference between them is attributable to the hint and nothing else.

**Which change matters more.** Changing the prompt rewrote the entire image except the posture.
Changing the hint rewrote only the posture. That asymmetry is the finding: **the skeleton owns where
the body is, the prompt owns what the body is.**

One honest wrinkle. The same prompt asked for a short dress with bare legs, and the seated pose got
one while the standing pose was given a longer dress that covered the calves and hid the feet. The
worst joint in `output_02.png` is the knee, at 0.144, and that is where the fabric is. Garment length
is not reliably controllable from the prompt, and where clothing covers a joint the score degrades
because the re-detected keypoint has to be inferred.

### What the settings are, and why

Each of these came from a controlled comparison with everything else held constant.

| Setting | Evidence |
|---|---|
| **Draw the hint at the generation resolution** | `OpenposeDetector.__call__` renders at its own internal resolution and rescales the result: 832x1216 in, 832x**1280** out. Rescaling back blurred the stick figure from ~43 distinct colours to 2210 and squashed the body 5% vertically, and ControlNet, unable to tell which shin belonged to which thigh, drew **three legs**. Taking keypoints from `detect_poses()` and drawing them once at the target size removed the extra limb with prompt, seed and scale unchanged. Hint sharpness is part of the conditioning signal, not cosmetics. |
| **Letterbox, do not centre-crop** | `ImageOps.fit` discarded 12% of the height of a 355x592 reference and took its left ankle out of frame: 18/18 keypoints down to 17/18. `ImageOps.pad` keeps the whole body. |
| **`conditioning_scale` 1.0, not 0.8** | At 0.8 a folded knee landed 0.448 of the frame from the hint (mean 0.099); at 1.0 every joint was within 0.066 (mean 0.031). The torso and arms matched at both scales, so the failure was invisible without measuring. |
| **Keep hand keypoints enabled** | Common guidance for this ControlNet says to disable hand detection. Measured with everything else held: the body-only hint scored mean 0.108 against 0.085 and slid the braced wrist 0.278 out of place. Hand keypoints anchor the wrist even though they do not fix the fingers. |
| **Choose placement from RAM as well as VRAM** | Offloading trades VRAM for system RAM, so on a RAM-limited VM it makes the failure that kills the session more likely, not less. A 15 GB T4 holds the whole pipeline, making the fast path also the safe one. |
| **`enable_vae_tiling()`, not `enable_vae_slicing()`** | Slicing splits a batch of latents; the diffusers docs state no performance impact for single-image batches, which is all this notebook generates. Tiling is what lowers peak memory for one large image. |

### Limitations observed

1. **A perfect skeleton is not sufficient.** A hint can score 18/18 and still fail if it is rendered
   at the wrong size or blurred. Detection quality and hint quality are separate problems.
2. **Detection failure is silent.** A broken hint does not produce a broken pose, it produces a
   *default* pose: with nothing coherent to enforce, ControlNet contributes nothing and the model
   falls back on its own prior. Nothing in the run reports it, which is why section 3a exists.
3. **Occlusion is what breaks detection.** An arm hidden inside a coat cost a whole limb chain
   (15/18); an upper-body crop had no leg keypoints at all (12/18).
4. **The hint is 2D.** No depth is encoded, so overlapping limbs cannot be disambiguated. Facing
   direction has to come from the prompt.
5. **Fingers on a raised hand come out merged.** Three hint variants were compared with everything
   else held constant - body plus hand keypoints, body only, and body only with thicker limb lines -
   and all three failed. Hands occupy too few pixels at 832x1216. An inpainting pass over the hand
   region is the real fix and is outside this tool's scope. Hands resting on a surface come out
   clearly better than raised ones, which is why `pose_02` uses closed fists.
6. **Clothing can hide the very thing being demonstrated.** A floor-length dress covered the legs
   completely, which made the pose unverifiable in the output and degraded the fidelity score because
   the ankles had to be guessed. Same hint and seed, the ink-drawing prompt scored 0.046 and the
   floor-length one 0.098. The garment now leaves the legs and feet visible.
7. **Framing is not independent of the pose.** The crop follows the skeleton's extent in frame.
8. **Prompt clauses are not honoured equally.** Concrete beats abstract - a clause naming a material
   and an optical effect is rendered, a bare abstract noun tends to be ignored. Each text encoder
   also truncates at 77 tokens silently, from the tail, so any prompt edit needs re-checking against
   the tokenizer.
9. **Single subject only.** Multi-person skeletons are detected but SDXL blends identities between
   overlapping figures. Not exercised here.
10. **Realism is SDXL 1.0's ceiling.** A photoreal fine-tune is a drop-in `BASE_ID` swap and would
    help; matching a frontier image model needs a different base model, not different settings.
11. **Not FLUX.** SDXL was chosen so this runs on a free T4. FLUX.2-klein is 9B with a 24B text
    encoder and has no official ControlNet; FLUX.1-dev has a usable pose ControlNet but needs a paid
    runtime plus quantisation.

### Reproducibility

Fixed seeds reproduce a comparison on the same GPU and library versions. Exact pixels are not
portable across GPUs or `diffusers` versions, since cuDNN kernel selection and fp16 accumulation
order differ. Every prompt, seed and setting is in `prompts.md`.